# The EM algorithm (general) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Alternate soft assignment and parameter refitting when data has hidden causes
EM maximizes likelihood with latent variables by iterating expected complete-data counts and parameter updates. These examples run one mixture-of-coins E-step, M-step, likelihood tracking, soft versus hard assignment, and convergence behavior.

In [ ]:
# 1) E-step responsibilities for coin-mixture sequences
heads=np.array([8,2,7,3]); tails=10-heads; pi=np.array([0.5,0.5]); theta=np.array([0.8,0.3])
like=np.vstack([theta[k]**heads*(1-theta[k])**tails for k in [0,1]]).T; resp=like*pi; resp=resp/resp.sum(1,keepdims=True)
plt.figure(figsize=(6,3)); plt.imshow(resp,aspect='auto'); plt.colorbar(); plt.title('soft responsibilities')
assert resp[0,0]>0.99 and resp[1,1]>0.99

In [ ]:
# 2) M-step weighted coin biases
Nk=resp.sum(0); theta_new=(resp*heads[:,None]).sum(0)/(resp*10).sum(0); pi_new=Nk/len(heads)
plt.figure(figsize=(6,3)); plt.bar(['theta A','theta B'],theta_new); plt.title('weighted refit')
assert theta_new[0]>theta_new[1]

In [ ]:
# 3) observed likelihood rises after the EM update
def ll(pi,theta): return sum(math.log(sum(pi[k]*(theta[k]**h)*((1-theta[k])**(10-h)) for k in [0,1])) for h in heads)
before=ll(pi,theta); after=ll(pi_new,theta_new)
plt.figure(figsize=(6,3)); plt.bar(['before','after'],[before,after]); plt.title('EM does not decrease likelihood')
assert after>=before

In [ ]:
# 4) soft counts differ from brittle hard assignments
hard=np.eye(2)[resp.argmax(1)]; th_h=(hard*heads[:,None]).sum(0)/(hard*10).sum(0)
plt.figure(figsize=(6,3)); plt.plot(theta_new,'o-',label='soft'); plt.plot(th_h,'x--',label='hard'); plt.legend(); plt.title('soft assignments retain uncertainty')
assert np.linalg.norm(theta_new-th_h)>0

In [ ]:
# 5) several EM iterations stabilize parameters
pi=np.array([0.5,0.5]); theta=np.array([0.6,0.4]); hist=[]
for _ in range(8):
    like=np.vstack([theta[k]**heads*(1-theta[k])**tails for k in [0,1]]).T; r=like*pi; r=r/r.sum(1,keepdims=True); pi=r.sum(0)/len(heads); theta=(r*heads[:,None]).sum(0)/(r*10).sum(0); hist.append(theta.copy())
plt.figure(figsize=(6,3)); plt.plot(np.array(hist)); plt.title('EM parameter path')
assert np.linalg.norm(hist[-1]-hist[-2])<0.02